In [ ]:
# JUSTIN

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
from multihead_model import MultiHeadModel
from config import D_TABULAR, D_EMBEDDING, D_FUSION

In [3]:
# Configure modality and model
# Select which modality to train: "tabular" or "image"
MODALITY = "image"   # change to "image" if you want image training
NUM_CLASSES = 5        # Diabetes_012 is binary classification (0/1)

In [4]:
import importlib, multihead_model
importlib.reload(multihead_model)
from multihead_model import MultiHeadModel

In [5]:

model = MultiHeadModel(
    d_tabular = D_TABULAR, 
    d_embedding = D_EMBEDDING, 
    d_fusion = D_FUSION,
    n_tabular_classes = None, 
    n_image_classes = NUM_CLASSES, 
    n_multi_classes = None
    )

device = "cuda" if torch.cuda.is_available() else "cpu"
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=0.0)
model = model.to(device)


In [6]:
# Build DataLoaders
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
import pandas as pd
from pathlib import Path
import math
import torch

In [7]:
# ----- Image dataset (returns dict for consistency) -----
class ImageFolderDict(datasets.ImageFolder):
    def __getitem__(self, index):
        img, label = super().__getitem__(index)
        return {"img": img, "label": torch.tensor(label, dtype=torch.long)}

In [13]:
import image_basic_preprocessing
importlib.reload(image_basic_preprocessing)
from image_basic_preprocessing import tfms

IMG_TFMS = tfms
TRAIN_DIR = Path("image_dataset/split/train")
VAL_DIR   = Path("image_dataset/split/val")

In [ ]:
def build_loaders(modality, batch_size=64, workers=4):
    if VAL_DIR.exists():
        train_ds = ImageFolderDict(TRAIN_DIR, transform=IMG_TFMS)
        val_ds   = ImageFolderDict(VAL_DIR,   transform=IMG_TFMS)
    else:
        full = ImageFolderDict(TRAIN_DIR, transform=IMG_TFMS)
        n = len(full); n_val = math.floor(0.2 * n)
        train_ds, val_ds = random_split(full, [n - n_val, n_val], generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True)
    return train_loader, val_loader

train_loader, val_loader = build_loaders(MODALITY, batch_size=64, workers=4)

# quick check
b = next(iter(train_loader))
print("Batch keys:", list(b.keys()))
for k,v in b.items():
    if isinstance(v, torch.Tensor):
        print(k, tuple(v.shape), v.dtype)

c:\Users\zheng\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
# Training and Evaluation loops
import torch.nn.functional as F
import torch

In [ ]:
def train_one_epoch(model, loader, optimizer, device, modality, log_every=50):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0

    for i, batch in enumerate(loader, 1):
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                batch[k] = v.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(task_type="image", x_img=batch["img"])
        loss = F.cross_entropy(logits, batch["label"])
        loss.backward()
        optimizer.step()

        bs = batch["label"].size(0)
        loss_sum += loss.item() * bs
        total    += bs
        correct  += (logits.argmax(1) == batch["label"]).sum().item()

        if i % log_every == 0:
            print(f"  step {i:4d} | loss {loss_sum/max(total,1):.4f} | acc {correct/max(total,1):.4f}")

    return {"loss": loss_sum/max(total,1), "acc": correct/max(total,1)}

@torch.no_grad()
def evaluate(model, loader, device, modality):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0

    for batch in loader:
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                batch[k] = v.to(device, non_blocking=True)

        logits = model(task_type="image", x_img=batch["img"])
        loss = F.cross_entropy(logits, batch["label"])
        bs = batch["label"].size(0)
        loss_sum += loss.item() * bs
        total    += bs
        correct  += (logits.argmax(1) == batch["label"]).sum().item()

    return {"loss": loss_sum/max(total,1), "acc": correct/max(total,1)}

In [ ]:
# Main training loop
import os, copy

def run_training(model, train_loader, val_loader, modality, epochs=10, save_dir="runs/exp_single"):
    os.makedirs(save_dir, exist_ok=True)
    best_ckpt = os.path.join(save_dir, "best.pth")

    best_val = float("inf")
    best_state = None

    for epoch in range(1, epochs+1):
        print(f"\nEpoch {epoch}/{epochs}")
        tr = train_one_epoch(model, train_loader, optimizer, device, modality, log_every=50)
        va = evaluate(model, val_loader, device, modality)

        print(f"train: loss={tr['loss']:.4f}, acc={tr['acc']:.4f} | "
              f"val: loss={va['loss']:.4f}, acc={va['acc']:.4f}")

        if va["loss"] < best_val - 1e-6:
            best_val = va["loss"]
            best_state = {
                "epoch": epoch,
                "model": copy.deepcopy(model.state_dict()),
                "optimizer": optimizer.state_dict(),
                "val_loss": best_val,
                "modality": modality
            }
            torch.save(best_state, best_ckpt)
            print(f"[best updated] → {best_ckpt}")

# Example run (only one modality at a time)
run_training(model, train_loader, val_loader, MODALITY, epochs=5, save_dir=f"runs/{MODALITY}_exp")
